In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader
import random
import numpy as np

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------
# Siamese Network Definition
# -----------------------------
class SiameseNetwork(nn.Module):
    def __init__(self):
        super(SiameseNetwork, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, 3),  # 28x28 -> 26x26
            nn.ReLU(),
            nn.MaxPool2d(2),      # 26x26 -> 13x13
            nn.Conv2d(16, 32, 3), # 13x13 -> 11x11
            nn.ReLU(),
            nn.MaxPool2d(2)       # 11x11 -> 5x5
        )
        self.fc = nn.Sequential(
            nn.Linear(32 * 5 * 5, 128),
            nn.ReLU(),
            nn.Linear(128, 64)
        )

    def forward_once(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

    def forward(self, x1, x2):
        out1 = self.forward_once(x1)
        out2 = self.forward_once(x2)
        return out1, out2

# -----------------------------
# Custom Siamese Dataset
# -----------------------------
class SiameseMNIST(Dataset):
    def __init__(self, mnist):
        self.mnist = mnist
        self.data = mnist.data
        self.targets = mnist.targets

        self.class_indices = {}
        for i, label in enumerate(self.targets):
            label = label.item()
            if label not in self.class_indices:
                self.class_indices[label] = []
            self.class_indices[label].append(i)

    def __getitem__(self, index):
        img1, label1 = self.data[index], self.targets[index].item()
        should_match = random.randint(0, 1)

        if should_match:
            idx2 = random.choice(self.class_indices[label1])
        else:
            label2 = random.choice([l for l in self.class_indices if l != label1])
            idx2 = random.choice(self.class_indices[label2])

        img2 = self.data[idx2]
        label = torch.tensor([int(label1 != self.targets[idx2])], dtype=torch.float32)
        return img1.unsqueeze(0).float() / 255.0, img2.unsqueeze(0).float() / 255.0, label
    def __len__(self):
        return len(self.mnist)

# -----------------------------
# Training the Siamese Network
# -----------------------------
def train(model, loader, optimizer, criterion, epochs=5):
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for img1, img2, label in loader:
            img1, img2, label = img1.to(device), img2.to(device), label.to(device)

            out1, out2 = model(img1, img2)
            loss = criterion(out1, out2, label)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
        print(f"Epoch {epoch + 1}, Loss: {running_loss / len(loader):.4f}")

# -----------------------------
# Contrastive Loss
# -----------------------------
class ContrastiveLoss(nn.Module):
    def __init__(self, margin=1.0):
        super().__init__()
        self.margin = margin

    def forward(self, out1, out2, label):
        euclidean_distance = F.pairwise_distance(out1, out2)
        return torch.mean((1 - label) * euclidean_distance.pow(2) +
                          label * torch.clamp(self.margin - euclidean_distance, min=0).pow(2))

# -----------------------------
# Load MNIST and Train
# -----------------------------
transform = transforms.ToTensor()
mnist_train = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
mnist_test = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_dataset = SiameseMNIST(mnist_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

model = SiameseNetwork().to(device)
criterion = ContrastiveLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

train(model, train_loader, optimizer, criterion, epochs=5)

# -----------------------------
# Create Support Set (1 image per digit)
# -----------------------------
support_set = {}
for img, label in mnist_train:
    if label not in support_set:
        support_set[label] = img.unsqueeze(0).to(device)
    if len(support_set) == 10:
        break

# -----------------------------
# Test on 100 Images
# -----------------------------
model.eval()
correct = 0
for i in range(100):
    test_img, true_label = mnist_test[i]
    test_img = test_img.unsqueeze(0).to(device)

    min_dist = float("inf")
    predicted = -1
    for digit, ref_img in support_set.items():
        with torch.no_grad():
            emb1 = model.forward_once(test_img)
            emb2 = model.forward_once(ref_img)
            dist = F.pairwise_distance(emb1, emb2)
        if dist.item() < min_dist:
            min_dist = dist.item()
            predicted = digit

    if predicted == true_label:
        correct += 1

accuracy = correct / 100
print(f"\n✅ One-shot classification accuracy (on 100 samples): {accuracy * 100:.2f}%")

Epoch 1, Loss: 0.2578
Epoch 2, Loss: 0.2555
Epoch 3, Loss: 0.2543
Epoch 4, Loss: 0.2541
Epoch 5, Loss: 0.2540

✅ One-shot classification accuracy (on 100 samples): 12.00%
